# Weighted Lebesgue Spaces: Flat vs Radial Laplacian

The weight $w(r) = r^2$ arises naturally in spherical coordinates: the radial inner product is
$\langle u, v \rangle_{r^2} = \int u(r)\, v(r)\, r^2\, dr$.

The natural operator on this space is the **radial Laplacian**
$-\frac{d^2}{dr^2} - \frac{2}{r}\frac{d}{dr}$,
which is self-adjoint w.r.t. $\langle\cdot,\cdot\rangle_{r^2}$.
Its Dirichlet eigenfunctions are $\varphi_n(r) \propto \sin(n\pi r/R)/r$ — qualitatively
different from the flat $\sin(n\pi r/R)$.

This notebook demonstrates:
1. How `WeightedLebesgue` implements $L^2(r^2)$ via `MassWeightedHilbertSpace`
2. The different eigenfunctions of the flat vs radial Laplacian
3. How `BesselSobolevInverse` as a prior looks different in each geometry

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from intervalinf import IntervalDomain, Function
from intervalinf.spaces import Lebesgue, WeightedLebesgue
from intervalinf.core.boundary import BoundaryConditions
from intervalinf.core.config import IntegrationConfig
from intervalinf.operators import Laplacian, RadialLaplacian, BesselSobolevInverse

print("Imports OK")

## 1. Spaces and Operators

In [ ]:
domain = IntervalDomain(0.0, 1.0)
bc = BoundaryConditions.dirichlet()
cfg = IntegrationConfig(method="simpson", n_points=4000)
N = 20

# Plain L²([0,1]) + flat Laplacian  (-d²/dr²)
space_flat = Lebesgue(N, domain, basis=None, integration_config=cfg)
L_flat = Laplacian(space_flat, bc, 1.0, method="spectral", dofs=N, integration_config=cfg)

# Weighted L²([0,1]; r²) + radial Laplacian  (-d²/dr² - 2/r d/dr)
space_radial = WeightedLebesgue(N, domain, lambda r: np.asarray(r)**2, integration_config=cfg)
L_radial = RadialLaplacian(space_radial, bc, 1.0, method="spectral", dofs=N, integration_config=cfg)

print("Flat    eigenvalues (first 3):", [f"{L_flat.get_eigenvalue(j):.3f}"   for j in range(3)])
print("Radial  eigenvalues (first 3):", [f"{L_radial.get_eigenvalue(j):.3f}" for j in range(3)])

## 2. Eigenfunctions: Flat vs Radial Laplacian

Both operators have the same Dirichlet spectrum $\lambda_n = (n\pi)^2$ on $[0,1]$, but
their eigenfunctions are qualitatively different:
- **Flat:** $\varphi_n(r) = \sqrt{2}\,\sin(n\pi r)$ — bounded, symmetric bumps
- **Radial:** $\varphi_n(r) = \sqrt{2}\,\sin(n\pi r)/r$ — peaks sharply near the origin due to the $1/r$ factor

In [ ]:
r = np.linspace(0.01, 0.99, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['C0', 'C1', 'C2']
for j in range(3):
    lam_f = L_flat.get_eigenvalue(j)
    lam_r = L_radial.get_eigenvalue(j)
    phi_f = np.array([L_flat.get_eigenfunction(j)(x) for x in r])
    phi_r = np.array([L_radial.get_eigenfunction(j)(x) for x in r])

    axes[0].plot(r, phi_f, color=colors[j], linewidth=2,
                 label=f'φ_{j}  λ={lam_f:.1f}')
    axes[1].plot(r, phi_r, color=colors[j], linewidth=2,
                 label=f'φ_{j}  λ={lam_r:.1f}')
    axes[2].plot(r, phi_r, color=colors[j], linestyle='--', linewidth=2)
    axes[2].plot(r, phi_f, color=colors[j], linewidth=2)

axes[0].set_title('Flat Laplacian on $L^2$\neigenfns $\\sim \\sin(n\\pi r)$')
axes[1].set_title('Radial Laplacian on $L^2(r^2)$\neigenfns $\\sim \\sin(n\\pi r)/r$')
axes[2].set_title('Overlay: flat (solid) vs radial (dashed)')

for ax in axes:
    ax.set_xlabel('r')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Bessel-Sobolev Prior: Flat vs Radial

In [ ]:
k, s = 2.0, 1.0

C_flat   = BesselSobolevInverse(space_flat,   space_flat,   k, s, L_flat,   dofs=N, integration_config=cfg)
C_radial = BesselSobolevInverse(space_radial, space_radial, k, s, L_radial, dofs=N, integration_config=cfg)

print(f"Flat   prior:  fast={C_flat._can_use_fast_transforms}")
print(f"Radial prior:  radial_fast={C_radial._radial_dirichlet_fast}")

## 4. Prior Covariance Kernel

In [ ]:
# Covariance kernel: C(r, r0) = (C δ_{r0})(r), approximated by applying C to a narrow bump at r0.
# Shows how the prior "spreads" information differently in flat vs radial geometry.

r0_values = [0.25, 0.5, 0.75]
r_eval = np.linspace(0.02, 0.98, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r0 in r0_values:
    bump = Function(domain, evaluate_callable=lambda x, r0=r0: np.exp(-((np.asarray(x) - r0)**2) / (2*0.02**2)))
    Cf_flat   = C_flat(bump)
    Cf_radial = C_radial(bump)
    fv_flat   = np.array([Cf_flat(x)   for x in r_eval])
    fv_radial = np.array([Cf_radial(x) for x in r_eval])
    axes[0].plot(r_eval, fv_flat,   linewidth=2, label=f'r₀={r0}')
    axes[1].plot(r_eval, fv_radial, linewidth=2, label=f'r₀={r0}')

axes[0].set_title(f'Flat prior  C=(k²–Δ)⁻¹  k={k}, s={s}')
axes[1].set_title(f'Radial prior C=(k²–Δᵣ)⁻¹  k={k}, s={s}')
for ax in axes:
    ax.set_xlabel('r')
    ax.set_ylabel('(C bump)(r)')
    ax.legend()
    ax.axhline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

- **`WeightedLebesgue(r²)`** implements $L^2(r^2)$ via a mass operator — the natural space for spherically symmetric problems
- The **radial Laplacian** is self-adjoint on $L^2(r^2)$; its eigenfunctions are $\sim \sin(n\pi r)/r$, not $\sin(n\pi r)$
- The **flat Laplacian** is self-adjoint on plain $L^2$; using it on a weighted space would break self-adjointness
- The `BesselSobolevInverse` prior inherits this geometry: radial priors spread differently near the origin

In [ ]:
r_fine = np.linspace(0.12, 0.98, 200)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Eigenvalues comparison
eigenvalues_plain = [L_plain.get_eigenvalue(j) for j in range(10)]
eigenvalues_weighted = [L_weighted.get_eigenvalue(j) for j in range(10)]

axes[0, 0].semilogy(range(10), eigenvalues_plain, 'bo-', linewidth=2, markersize=8, label='Plain L²')
axes[0, 0].semilogy(range(10), eigenvalues_weighted, 'rs--', linewidth=2, markersize=8, label='Weighted L²(r²)')
axes[0, 0].set_xlabel('Mode index j')
axes[0, 0].set_ylabel('Eigenvalue λ_j')
axes[0, 0].set_title('Laplacian Eigenvalues')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# First three eigenfunctions (plain)
axes[0, 1].set_title('Eigenfunctions on Plain L²')
for j in range(3):
    phi_j = L_plain.get_eigenfunction(j)
    phi_vals = np.array([phi_j(r) for r in r_fine])
    axes[0, 1].plot(r_fine, phi_vals, linewidth=2, label=f'φ₀ (λ={eigenvalues_plain[j]:.1f})')
axes[0, 1].set_xlabel('r')
axes[0, 1].set_ylabel('φ_j(r)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# First three eigenfunctions (weighted)
axes[1, 0].set_title('Eigenfunctions on Weighted L²(r²)')
for j in range(3):
    phi_j = L_weighted.get_eigenfunction(j)
    phi_vals = np.array([phi_j(r) for r in r_fine])
    axes[1, 0].plot(r_fine, phi_vals, linewidth=2, label=f'φ_{j} (λ={eigenvalues_weighted[j]:.1f})')
axes[1, 0].set_xlabel('r')
axes[1, 0].set_ylabel('φ_j(r)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Orthonormality check: norms w.r.t. each space's inner product
plain_norms = [space_plain.inner_product(L_plain.get_eigenfunction(j), L_plain.get_eigenfunction(j)) for j in range(6)]
weighted_norms = [space_weighted.inner_product(L_weighted.get_eigenfunction(j), L_weighted.get_eigenfunction(j)) for j in range(6)]

x = np.arange(6)
width = 0.35
axes[1, 1].bar(x - width/2, plain_norms, width, label='Plain L²', color='b', alpha=0.7)
axes[1, 1].bar(x + width/2, weighted_norms, width, label='Weighted L²(r²)', color='r', alpha=0.7)
axes[1, 1].set_xlabel('Mode index j')
axes[1, 1].set_ylabel('‖φ_j‖² (w.r.t. space inner product)')
axes[1, 1].set_title('Eigenfunction Normalization')
axes[1, 1].set_xticks(x)
axes[1, 1].legend()
axes[1, 1].axhline(y=1.0, color='k', linestyle='--', alpha=0.3)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Eigenvalues (first 5 modes):")
print(f"  j=0: plain={eigenvalues_plain[0]:.4f}, weighted={eigenvalues_weighted[0]:.4f}")
print(f"  j=1: plain={eigenvalues_plain[1]:.4f}, weighted={eigenvalues_weighted[1]:.4f}")
print(f"  j=2: plain={eigenvalues_plain[2]:.4f}, weighted={eigenvalues_weighted[2]:.4f}")
print(f"\n✅ Note: eigenvalues are identical — the Laplacian spectrum is unchanged by weighting")